## Scope
Included: text-id, answer **correctness**, answer-screen region labels, per-area
dwell/skip/fixation/pupil metrics, last-visited area, **fixation-sequence tags**
(+ simplified sequences / visit counts), and **answer-region RT/TFD** (computed
without a paragraph screen). **Skipped:** button-click-based run RT and
last-select/confirm labels (the button-clicks builder will be adapted to this
dataset later).


## 1. Config

In [9]:
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(6):
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
        p = p.parent
    return start.resolve()

PROJECT_ROOT = _find_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import ast
import pandas as pd
from src import constants as C
from src.data_prep import data_csv_generation as G
from src.derived.pupil_norm import compute_participant_pupil_stats

DIANA2_DIR = PROJECT_ROOT / "data" / "Experiment" / "fake results" / "Diana2"
IA_ANSWERS_PATH  = DIANA2_DIR / "ia_answers.csv"
FIX_ANSWERS_PATH = DIANA2_DIR / "fix_answers.csv"

PARTICIPANT_SRC = "RECORDING_SESSION_LABEL"   # -> participant_id
ID_TO_LETTER = {0: "A", 1: "B", 2: "C", 3: "D"}

OUTPUT_DIR = PROJECT_ROOT / "data" / "new_exp_try_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "all_participants.csv"
print("output ->", OUTPUT_PATH)


output -> E:\Technion\QA PROJECT\programming\QA_eyetracking_workspace\data\new_exp_try_runs\all_participants.csv


## 2. Load & adapt to the pipeline schema
`load_report` reads a UTF-8 CSV report and renames the session label to
`participant_id`. `adapt_answers` converts the raw answer format and fills the
few derived columns the base features read.

**Correctness mapping (do not mix up correctness vs. on-screen order):**
- `answers_order[screen_pos] = canonical answer id` (0=A…3=D); we convert it to
  letters for the label features.
- `FINAL_ANSWER` = the **on-screen position** the participant selected →
  `selected_answer_position`.
- `correct_answer` (from the experiment design) = the **on-screen position** of
  the correct option (answer_A is always correct in OneStopQA) →
  `correct_answer_position` **directly**.
- `is_correct = (selected_answer_position == correct_answer_position)` — both are
  on-screen positions, so they are directly comparable.


In [10]:
def load_report(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, low_memory=False).rename(
        columns={PARTICIPANT_SRC: C.PARTICIPANT_ID}
    )


def adapt_answers(df: pd.DataFrame) -> pd.DataFrame:
    """Raw EyeLink answer-screen export -> schema the L1 base features expect."""
    df = df.copy()

    # raw integer answers_order: screen position -> canonical answer id (0=a..3=d)
    int_order = df[C.ANSWERS_ORDER_COLUMN].apply(
        lambda x: x if isinstance(x, list) else ast.literal_eval(x)
    )

    # answer-correctness positions (on-screen positions 0..3):
    #   selected = FINAL_ANSWER  (chosen on-screen location; -1 if unanswered)
    #   correct  = `correct_answer`, which is ALREADY the on-screen position of
    #              the correct option. In build_exp_inputs.ipynb it is defined as
    #              `inverse_order[0]` = "where answer_A moved" (answer_A is always
    #              the correct OneStopQA option), i.e. the screen slot the correct
    #              answer landed in. So it is used directly here -- it must NOT be
    #              re-looked-up through answers_order (that double-applies the
    #              permutation and corrupts is_correct).
    df[C.SELECTED_ANSWER_POSITION_COLUMN] = df["FINAL_ANSWER"].astype(int)
    df[C.CORRECT_ANSWER_POSITION_COLUMN] = df["correct_answer"].astype(int)

    # integer answers_order -> letters ['C','B','A','D'] (0->A,1->B,2->C,3->D)
    df[C.ANSWERS_ORDER_COLUMN] = int_order.apply(
        lambda o: str([ID_TO_LETTER[int(i)] for i in o])
    )

    # 0-indexed screen answers answer_0..3 -> 1-indexed answer_1..4
    for i in (3, 2, 1, 0):
        df[f"answer_{i + 1}"] = df[f"answer_{i}"]

    # canonical (correctness-level) answers already exist in the raw export as
    # answer_a..d -> just rename to the uppercase answer_A..D the pipeline uses.
    # These are byte-identical to add_answer_text_columns' location-based build,
    # so that base feature is dropped from BASE_FUNCS.
    df = df.rename(
        columns={f"answer_{lab.lower()}": f"answer_{lab}" for lab in C.ANSWER_LABELS}
    )

    # derived / renamed columns the base features read
    df[C.PRACTICE_TRIAL_COLUMN] = df["practice"].astype(bool)
    df[C.REPEATED_TRIAL_COLUMN] = False                       # no repeated-reading here
    df[C.SAME_CRITICAL_SPAN_COLUMN] = df["onestopqa_question_id"]
    return df


REMOVE_PRACTICE = True    # pipeline convention: drop practice trials
DROP_UNANSWERED = True    # FINAL_ANSWER == -1 are recalibration trials
ia = adapt_answers(load_report(IA_ANSWERS_PATH))
if REMOVE_PRACTICE:
    ia = ia[~ia[C.PRACTICE_TRIAL_COLUMN]].copy()
if DROP_UNANSWERED:
    ia = ia[ia[C.SELECTED_ANSWER_POSITION_COLUMN] >= 0].copy()
fix = load_report(FIX_ANSWERS_PATH)
print("adapted ia_answers:", ia.shape, "| participants:", list(ia[C.PARTICIPANT_ID].unique()))


adapted ia_answers: (570, 331) | participants: ['D_Test2']


## 3. Choose the reduced feature set

In [11]:
BASE_FUNCS = [
    "add_text_id",
    "add_text_id_with_q",
    "add_is_correct",                # FINAL_ANSWER == correct answer position
    # "add_answer_text_columns" skipped: answer_A..D already come pre-built in the
    # raw export (answer_a..d) and are renamed in adapt_answers.
    "add_IA_screen_location",
    "add_IA_answer_label",
    "add_selected_answer_label",     # letter of the selected answer
    "add_total_answering_RT_normalized",
    "add_zscored_pupil_columns",
]
GROUP_FUNCS = [
    "create_mean_area_dwell_time",
    "create_mean_area_fix_count",
    "create_mean_first_fix_duration",
    "create_skip_rate",
    "create_dwell_proportions",
    "create_mean_pupil_size_metrics",
    "create_first_encounter_pupil_size",
    "create_last_area_and_location_visited",
]
# Fixation-sequence tags + answer-region RT/TFD are added in the run cell below
# (they need special wiring: an in-memory fixations frame / no-paragraph mode).
# Still skipped: button-click-based run RT and last-select/confirm labels
# (button_clicks builder to be adapted to this dataset later).


## 4. Run base + group features
Participant pupil stats are computed from `fix_answers.tsv` and injected into the
pupil z-scoring feature (the same wiring `main()` uses).

In [12]:
import functools
from src.derived.reading_times import build_rt_and_tfd

pupil_stats = compute_participant_pupil_stats(fix)

# --- base features ---
base = G.resolve_base_functions(BASE_FUNCS)
base = [
    (fn, {**kw, "pupil_stats": pupil_stats}) if fn is G.add_zscored_pupil_columns else (fn, kw)
    for fn, kw in base
]
out = G.add_base_features(ia, base, verbose=True)

# --- group features: area-level, then fixation-sequence tags ---
group = G.resolve_group_functions(GROUP_FUNCS)

def fix_seq_tags(df):  # passes the in-memory answer-screen fixations frame
    return G.create_fixation_sequence_tags(df, fix_path=fix)

group += [
    (fix_seq_tags, {"join_columns": [C.TRIAL_ID, C.PARTICIPANT_ID]}),
    (G.create_simplified_fixation_tags, {"join_columns": [C.TRIAL_ID, C.PARTICIPANT_ID]}),
    (G.create_simplified_visit_counts,
     {"join_columns": [C.TRIAL_ID, C.PARTICIPANT_ID, C.AREA_LABEL_COLUMN]}),
]
out = G.generate_new_row_features(group, out, verbose=True)

# --- answer-region RT/TFD (no paragraph screen; no button clicks in this test) ---
rt = build_rt_and_tfd(
    all_participants=out,
    include_paragraph=False,
    include_run_based_rt=False,
    save=False,
    verbose=True,
)
out = out.merge(
    rt.drop_duplicates([C.PARTICIPANT_ID, C.TRIAL_ID]),
    on=[C.PARTICIPANT_ID, C.TRIAL_ID],
    how="left",
)
print("final:", out.shape)


Running: add_text_id
Running: add_text_id_with_q
Running: add_is_correct
Running: add_IA_screen_location
Running: add_IA_answer_label
Running: add_selected_answer_label
Running: add_total_answering_RT_normalized
Running: add_zscored_pupil_columns
Running group feature: create_mean_area_dwell_time
Running group feature: create_mean_area_fix_count
Running group feature: create_mean_first_fix_duration
Running group feature: create_skip_rate
Running group feature: create_dwell_proportions
Running group feature: create_mean_pupil_size_metrics
Running group feature: create_first_encounter_pupil_size
Running group feature: create_last_area_and_location_visited
Running group feature: fix_seq_tags
Running group feature: create_simplified_fixation_tags
Running group feature: create_simplified_visit_counts
Computing answer-region RT/TFD...
final: (570, 396)


## 5. Save & summarize

In [13]:
out.to_csv(OUTPUT_PATH, index=False)
print("saved", out.shape, "->", OUTPUT_PATH)
print()
print("trials:", out[C.TRIAL_ID].nunique(), "| rows:", len(out))
print(out[C.AREA_LABEL_COLUMN].value_counts(dropna=False))

saved (570, 396) -> E:\Technion\QA PROJECT\programming\QA_eyetracking_workspace\data\new_exp_try_runs\all_participants.csv

trials: 12 | rows: 570
area_label
question    140
answer_B    113
answer_C    110
answer_D    105
answer_A    102
Name: count, dtype: int64


## 6. Model transfer test (train on L1, test on new data)

Trains the trial-level logistic-regression **answer-correctness** model on the L1
prepared features (`ready_all_features.csv`, the original experiment) and
evaluates it on the Diana2 trial features built from `out` above. Only features
present in **both** datasets are used — the new data lacks the button-click
last-select/confirm and run-based `TimeSinceOffset` features, which are excluded.


In [14]:
from src.constants import TRIAL_ID_COLS
from src.data_paths import READY_ALL_FEATURES_PATH, ALL_PARTICIPANTS_PROCESSED_PATH
from src.predictive_modeling.answer_correctness.model_data import (
    load_all_features, save_all_features, build_trial_level_model_df,
)
from src.predictive_modeling.answer_correctness.models.logreg_model import TrialLevelLogRegModel
from src.predictive_modeling.answer_correctness.evaluation_core import (
    evaluate_models_on_prepared_split,
)
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score, f1_score, roc_auc_score,
)

# --- train features: L1 prepared features (original experiment) ---
if Path(READY_ALL_FEATURES_PATH).exists():
    trial_train = load_all_features()
else:  # build + cache from the original L1 all_participants if not present yet
    trial_train = save_all_features(
        pd.read_csv(ALL_PARTICIPANTS_PROCESSED_PATH, low_memory=False)
    )

# --- test features: new data (read back the saved CSV so the sequence columns
# are in the string form the trial-feature builders expect) ---
new_saved = pd.read_csv(OUTPUT_PATH, low_memory=False)
trial_test = build_trial_level_model_df(
    df=new_saved,
    include_last_lbl_before_confirm_features=False,
    include_last_lbl_before_select_features=False,
)
print("train trials:", trial_train.shape, "| test trials:", trial_test.shape)


train trials: (19436, 200) | test trials: (12, 152)


In [15]:
# Features present in BOTH tables (numeric, non-constant on the test side).
non_feature = set(TRIAL_ID_COLS) | {C.IS_CORRECT_COLUMN, C.TEXT_ID_WITH_Q_COLUMN}
feature_cols = [
    c for c in trial_test.columns
    if c not in non_feature
    and c in trial_train.columns
    and pd.api.types.is_numeric_dtype(trial_train[c])
    and trial_test[c].notna().any()
]
print("shared features used:", len(feature_cols))

res = evaluate_models_on_prepared_split(
    models=[TrialLevelLogRegModel()],
    train_df=trial_train,
    test_df=trial_test,
    target_col=C.IS_CORRECT_COLUMN,
    feature_cols=feature_cols,
)["trial_level_log_reg"]

yt, yp, ypr = res.y_true, res.y_pred, res.y_prob
metrics = pd.Series({
    "n_test": res.n_test,
    "positives": res.n_positive,
    "negatives": res.n_negative,
    "accuracy": accuracy_score(yt, yp),
    "balanced_accuracy": balanced_accuracy_score(yt, yp),
    "macro_f1": f1_score(yt, yp, average="macro"),
    "roc_auc": roc_auc_score(yt, ypr) if len(set(yt.tolist())) > 1 else float("nan"),
})
print(metrics.round(3).to_string())


shared features used: 148
n_test               12.000
positives             5.000
negatives             7.000
accuracy              0.750
balanced_accuracy     0.786
macro_f1              0.748
roc_auc               0.971


In [16]:
# Per-trial predictions
pred_df = trial_test[list(TRIAL_ID_COLS)].copy()
pred_df["y_true"] = yt
pred_df["y_prob"] = ypr.round(3)
pred_df["y_pred"] = yp
pred_df


,participant_id,TRIAL_INDEX,y_true,y_prob,y_pred
0,D_Test2,2,0,0.578,1
1,D_Test2,5,0,0.829,1
2,D_Test2,6,0,0.102,0
3,D_Test2,7,0,0.107,0
4,D_Test2,10,0,0.581,1
5,D_Test2,11,1,0.998,1
6,D_Test2,12,0,0.261,0
7,D_Test2,13,0,0.015,0
8,D_Test2,16,1,0.880,1
9,D_Test2,17,1,0.723,1
